# 03.2 — GenRec Log-Likelihood Evaluation (Fair Comparison)

**RecSys 2026 Tutorial**: Choosing Between Explainable, Retrieval-Augmented, and LLM-Native Recommenders

## Why this notebook exists

The default GenRec evaluation (notebook 03/03.1) **generates** a single title per user
and checks if it matches the ground truth. But BPR-MF and RAG **score all 100 pool items**
and rank them. This is not an apples-to-apples comparison.

This notebook fixes that by computing the fine-tuned model's **log-likelihood** for each
pool item's title given the user's history. Each pool item gets a score, producing a
ranked list of 100 items — the same evaluation structure as the other two paradigms.

**Requirements**: Colab A100 (loads the QLoRA adapter from notebook 03.1)

**Input**: Trained adapter from `03_1_genrec_qlora_finetuning.ipynb`

**Output**: `generative_finetuned_results.json` with pooled ranking metrics comparable to Paradigms 1 and 2

## 0. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets torch

In [ ]:
TEST_MODE = True

if TEST_MODE:
    N_USERS = 20
    print('*** TEST MODE: 20 users ***')
else:
    N_USERS = None  # all users
    print('*** FULL MODE ***')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/Foundations of Large Language Models/Final Project'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
ADAPTER_DIR = f'{DRIVE_ROOT}/results/genrec_qlora/final_adapter'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), \
    f'No adapter found at {ADAPTER_DIR}. Run notebook 03.1 first.'
print(f'Adapter found at {ADAPTER_DIR}')

## 1. Load data and model

In [ ]:
import json
import pickle
import numpy as np

with open(f'{DATA_DIR}/shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
with open(f'{DATA_DIR}/genrec_test.json') as f:
    genrec_test = json.load(f)

item_titles = shared['item_titles']
test_ground_truth = shared['test_ground_truth']
candidate_pools = shared['candidate_pools']

print(f'Test users: {len(test_ground_truth)}')
print(f'Catalog items: {len(item_titles)}')
print(f'Pool size: {len(next(iter(candidate_pools.values())))} items/user')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print(f'Model loaded with adapter. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Log-likelihood scoring

For each user and each candidate item in the shared pool:
1. Build the prompt: `{instruction}\n\n### input:\n{history}\n\n### Response:\n`
2. Append the candidate item's title
3. Run a forward pass and compute the mean log-probability of the title tokens
4. Rank all 100 pool items by log-likelihood

This is a forward pass (not autoregressive generation), so it's faster and deterministic.

In [ ]:
import torch.nn.functional as F


def score_title(prompt_text, title_text):
    """Compute mean log-probability of title tokens given the prompt."""
    full_text = prompt_text + title_text
    prompt_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids']
    full_ids = tokenizer(full_text, return_tensors='pt')['input_ids'].to(model.device)

    prompt_len = prompt_ids.shape[1]
    title_len = full_ids.shape[1] - prompt_len

    if title_len <= 0:
        return -float('inf')

    with torch.no_grad():
        outputs = model(input_ids=full_ids)
        # logits shape: (1, seq_len, vocab_size)
        logits = outputs.logits[0]  # (seq_len, vocab_size)

    # For each title token position, get the log-prob of the actual token
    # logits[i] predicts token[i+1], so we look at positions prompt_len-1 to end-1
    title_logits = logits[prompt_len - 1 : -1]  # (title_len, vocab_size)
    title_targets = full_ids[0, prompt_len:]     # (title_len,)

    log_probs = F.log_softmax(title_logits, dim=-1)
    token_log_probs = log_probs.gather(1, title_targets.unsqueeze(1)).squeeze(1)

    # Mean log-prob (normalizes for title length)
    return token_log_probs.mean().item()


# Sanity check: score the ground truth vs a random title
sample = genrec_test[0]
prompt = f"{sample['instruction']}\n\n### input:\n{sample['input']}\n\n### Response:\n"
gt_title = sample['output']
random_title = 'Cooking for Beginners: Easy Recipes'

gt_score = score_title(prompt, gt_title)
rand_score = score_title(prompt, random_title)
print(f'Ground truth: "{gt_title}"  → score: {gt_score:.4f}')
print(f'Random title: "{random_title}"  → score: {rand_score:.4f}')
print(f'Ground truth scored {"higher" if gt_score > rand_score else "lower"} (expected: higher)')

## 3. Score all pool items for each user

In [ ]:
import time

# Build user_idx → test example mapping
test_by_user = {ex['user_idx']: ex for ex in genrec_test}

# Select users to evaluate
eval_users = list(candidate_pools.keys())
if N_USERS is not None:
    eval_users = eval_users[:N_USERS]

CHECKPOINT_PATH = f'{RESULTS_DIR}/genrec_likelihood_checkpoint.pkl'

# Resume from checkpoint if available
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    pool_predictions = ckpt['pool_predictions']
    latencies = ckpt['latencies']
    print(f'Resumed from checkpoint: {len(pool_predictions)} users done.')
else:
    pool_predictions = {}
    latencies = []

remaining = [u for u in eval_users if u not in pool_predictions]
print(f'Total: {len(eval_users)}, Remaining: {len(remaining)}')

for i, user_idx in enumerate(remaining):
    if user_idx not in test_by_user:
        continue

    example = test_by_user[user_idx]
    pool_items = candidate_pools[user_idx]
    prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"

    t0 = time.time()

    # Score each pool item's title
    scores = []
    for item_idx in pool_items:
        title = item_titles.get(item_idx, '')
        if not title:
            scores.append(-float('inf'))
            continue
        scores.append(score_title(prompt, title))

    # Rank by log-likelihood (highest first)
    ranked_indices = np.argsort(scores)[::-1]
    pool_predictions[user_idx] = [pool_items[j] for j in ranked_indices]

    latencies.append(time.time() - t0)

    if (i + 1) % 100 == 0:
        print(f'  {len(pool_predictions)}/{len(eval_users)} | '
              f'latency: {np.mean(latencies[-100:])*1000:.0f} ms/user')
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({'pool_predictions': pool_predictions, 'latencies': latencies}, f)

# Final save
with open(CHECKPOINT_PATH, 'wb') as f:
    pickle.dump({'pool_predictions': pool_predictions, 'latencies': latencies}, f)

print(f'\nDone. {len(pool_predictions)} users scored.')
print(f'Mean latency: {np.mean(latencies)*1000:.0f} ms/user ({len(candidate_pools[eval_users[0]])} items/user)')

## 4. Evaluate

In [ ]:
import math

def hit_at_k(ranked_list, ground_truth, k=10):
    return 1.0 if ground_truth in ranked_list[:k] else 0.0

def ndcg_at_k(ranked_list, ground_truth, k=10):
    for i, item in enumerate(ranked_list[:k]):
        if item == ground_truth:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_ranking(preds, gt, k_values=None):
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}
    for k in k_values:
        hits, ndcgs = [], []
        for uid, true_item in gt.items():
            if uid not in preds:
                continue
            hits.append(hit_at_k(preds[uid], true_item, k))
            ndcgs.append(ndcg_at_k(preds[uid], true_item, k))
        results[f'HR@{k}'] = np.mean(hits) if hits else 0.0
        results[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0.0
    return results


pool_ranking = evaluate_ranking(pool_predictions, test_ground_truth, k_values=[1, 5, 10, 20])

print(f'Shared-pool ranking — log-likelihood scoring ({len(pool_predictions)} users):')
for m, v in pool_ranking.items():
    print(f'  {m}: {v:.4f}')

# Cross-paradigm comparison
print('\n--- Cross-Paradigm Comparison (Shared Pool HR@10) ---')
print(f'{"Paradigm":<35} {"HR@10":>8} {"NDCG@10":>8}')
print('-' * 55)
print(f'{"RAG (FAISS + Qwen rerank)":<35} {0.2829:>8.4f} {0.1820:>8.4f}')
print(f'{"Explainable (BPR-MF)":<35} {0.2440:>8.4f} {0.1440:>8.4f}')
print(f'{"Generative (log-likelihood)":<35} {pool_ranking["HR@10"]:>8.4f} {pool_ranking["NDCG@10"]:>8.4f}')
print(f'{"Generative (title generation)":<35} {0.0445:>8.4f} {0.0445:>8.4f}')
print(f'{"Generative (zero-shot)":<35} {0.0150:>8.4f} {0.0150:>8.4f}')

## 5. Save results

In [ ]:
if TEST_MODE:
    print('*** TEST MODE: skipping save ***')
else:
    results = {
        'paradigm': 'generative_finetuned_likelihood',
        'model': f'QLoRA fine-tuned {MODEL_ID} (log-likelihood scoring)',
        'anchor_paper': 'GenRec (Ji et al., ECIR 2024)',
        'ranking_pooled': pool_ranking,
        'system': {
            'latency': {
                'mean_latency_ms': float(np.mean(latencies) * 1000),
                'p50_latency_ms': float(np.median(latencies) * 1000),
                'p95_latency_ms': float(np.percentile(latencies, 95) * 1000),
            },
            'scoring_method': 'log-likelihood',
            'pool_size': len(next(iter(candidate_pools.values()))),
        },
    }

    with open(f'{RESULTS_DIR}/generative_finetuned_likelihood_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    with open(f'{RESULTS_DIR}/generative_finetuned_likelihood_predictions.pkl', 'wb') as f:
        pickle.dump(pool_predictions, f)

    print(f'Results saved to {RESULTS_DIR}/')

In [ ]:
if not TEST_MODE:
    from google.colab import files
    files.download(f'{RESULTS_DIR}/generative_finetuned_likelihood_results.json')

## Summary

This notebook provides a **fair apples-to-apples comparison** for the generative paradigm
by scoring all 100 shared-pool items per user using log-likelihood, rather than generating
a single title and checking if it matches.

| Evaluation method | What it measures |
|---|---|
| **Title generation** (NB 03/03.1) | Can the model generate the exact correct title? |
| **Log-likelihood scoring** (this notebook) | Can the model rank the correct title above 99 negatives? |

The log-likelihood method is directly comparable to BPR-MF (dot-product scoring) and
RAG (cosine similarity scoring) — all three paradigms score the same 100 items and
produce a ranked list.